# Customer Data Processing using Apache Spark

## Objective

The objective of this project is to demonstrate Apache Spark DataFrame operations including:

- Data Cleaning
- Schema Management
- Data Transformation
- Filtering
- Aggregation
- GroupBy Operations
- Pivot Transformations
- ETL Pipeline Development

This project simulates a real-world customer analytics workflow using PySpark.

In [2]:
!pip install pyspark

In [10]:
# Initialize Spark environment

from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("CustomerAnalytics") \
    .getOrCreate()

In [11]:
# Create sample customer dataset

data = [
(101,"Rahul",25,"North",45000,"Male"),
(102,"Priya",28,"South",55000,"Female"),
(103,"Amit",35,"East",65000,"Male"),
(104,"Neha",22,"West",40000,"Female"),
(105,"Rohan",30,"North",70000,"Male"),
(106,"Anjali",27,"South",50000,"Female"),
(107,"Vikas",40,"East",85000,"Male"),
(108,"Pooja",24,"West",45000,"Female"),
(109,"Karan",32,"North",60000,"Male"),
(110,"Sneha",29,"South",58000,"Female"),
(111,None,26,"North",50000,"Male"),
(112,"Arjun",None,"West",48000,"Male"),
(113,"Kriti",31,None,62000,"Female"),
(114,"Rahul",25,"North",45000,"Male")
]

columns = ["Customer_ID","Name","Age","Region","Salary","Gender"]

# Create DataFrame
df = spark.createDataFrame(data, columns)

In [6]:
# Initial Dataset Preview

df.show()

+-----------+------+----+------+------+------+
|Customer_ID|  Name| Age|Region|Salary|Gender|
+-----------+------+----+------+------+------+
|        101| Rahul|  25| North| 45000|  Male|
|        102| Priya|  28| South| 55000|Female|
|        103|  Amit|  35|  East| 65000|  Male|
|        104|  Neha|  22|  West| 40000|Female|
|        105| Rohan|  30| North| 70000|  Male|
|        106|Anjali|  27| South| 50000|Female|
|        107| Vikas|  40|  East| 85000|  Male|
|        108| Pooja|  24|  West| 45000|Female|
|        109| Karan|  32| North| 60000|  Male|
|        110| Sneha|  29| South| 58000|Female|
|        111|  NULL|  26| North| 50000|  Male|
|        112| Arjun|NULL|  West| 48000|  Male|
|        113| Kriti|  31|  NULL| 62000|Female|
|        114| Rahul|  25| North| 45000|  Male|
+-----------+------+----+------+------+------+



In [12]:
# Inspect schema structure

df.printSchema()

root
 |-- Customer_ID: long (nullable = true)
 |-- Name: string (nullable = true)
 |-- Age: long (nullable = true)
 |-- Region: string (nullable = true)
 |-- Salary: long (nullable = true)
 |-- Gender: string (nullable = true)



In [13]:
# Check record count

print("Total Records:", df.count())

Total Records: 14


In [14]:
# Identify null values

from pyspark.sql.functions import *

df.select([
    count(when(col(c).isNull(), c)).alias(c)
    for c in df.columns
]).show()

+-----------+----+---+------+------+------+
|Customer_ID|Name|Age|Region|Salary|Gender|
+-----------+----+---+------+------+------+
|          0|   1|  1|     1|     0|     0|
+-----------+----+---+------+------+------+



In [15]:
# Replace missing values

df = df.fillna({
    "Name":"Unknown",
    "Age":0,
    "Region":"Unknown"
})

In [16]:
# Remove duplicate records

df = df.dropDuplicates()

In [17]:
# Rename salary column

df = df.withColumnRenamed(
    "Salary",
    "Monthly_Income"
)

In [18]:
# Create income category

df = df.withColumn(
    "Income_Category",
    when(col("Monthly_Income") >= 60000, "High")
    .otherwise("Medium")
)

In [19]:
# Filter customers above age threshold

df.filter(
    col("Age") > 25
).show()

+-----------+-------+---+-------+--------------+------+---------------+
|Customer_ID|   Name|Age| Region|Monthly_Income|Gender|Income_Category|
+-----------+-------+---+-------+--------------+------+---------------+
|        106| Anjali| 27|  South|         50000|Female|         Medium|
|        103|   Amit| 35|   East|         65000|  Male|           High|
|        105|  Rohan| 30|  North|         70000|  Male|           High|
|        107|  Vikas| 40|   East|         85000|  Male|           High|
|        102|  Priya| 28|  South|         55000|Female|         Medium|
|        113|  Kriti| 31|Unknown|         62000|Female|           High|
|        110|  Sneha| 29|  South|         58000|Female|         Medium|
|        109|  Karan| 32|  North|         60000|  Male|           High|
|        111|Unknown| 26|  North|         50000|  Male|         Medium|
+-----------+-------+---+-------+--------------+------+---------------+



In [20]:
# Filter high-income customers

df.filter(
    col("Monthly_Income") > 60000
).show()

+-----------+-----+---+-------+--------------+------+---------------+
|Customer_ID| Name|Age| Region|Monthly_Income|Gender|Income_Category|
+-----------+-----+---+-------+--------------+------+---------------+
|        103| Amit| 35|   East|         65000|  Male|           High|
|        105|Rohan| 30|  North|         70000|  Male|           High|
|        107|Vikas| 40|   East|         85000|  Male|           High|
|        113|Kriti| 31|Unknown|         62000|Female|           High|
+-----------+-----+---+-------+--------------+------+---------------+



In [21]:
# Generate summary metrics

df.select(
    count("*").alias("Total_Customers"),
    sum("Monthly_Income").alias("Total_Income"),
    avg("Monthly_Income").alias("Average_Income"),
    min("Monthly_Income").alias("Minimum_Income"),
    max("Monthly_Income").alias("Maximum_Income")
).show()

+---------------+------------+-----------------+--------------+--------------+
|Total_Customers|Total_Income|   Average_Income|Minimum_Income|Maximum_Income|
+---------------+------------+-----------------+--------------+--------------+
|             14|      778000|55571.42857142857|         40000|         85000|
+---------------+------------+-----------------+--------------+--------------+



In [22]:
# Analyze performance by region

df.groupBy("Region").agg(
    count("*").alias("Customer_Count"),
    avg("Monthly_Income").alias("Average_Income"),
    sum("Monthly_Income").alias("Total_Income")
).show()

+-------+--------------+------------------+------------+
| Region|Customer_Count|    Average_Income|Total_Income|
+-------+--------------+------------------+------------+
|Unknown|             1|           62000.0|       62000|
|  South|             3|54333.333333333336|      163000|
|   East|             2|           75000.0|      150000|
|   West|             3|44333.333333333336|      133000|
|  North|             5|           54000.0|      270000|
+-------+--------------+------------------+------------+



In [23]:
# Generate gender-wise income matrix

pivot_df = df.groupBy("Region") \
             .pivot("Gender") \
             .sum("Monthly_Income")

pivot_df.show()

+-------+------+------+
| Region|Female|  Male|
+-------+------+------+
|Unknown| 62000|  NULL|
|  South|163000|  NULL|
|   East|  NULL|150000|
|   West| 85000| 48000|
|  North|  NULL|270000|
+-------+------+------+



In [24]:
# Execute end-to-end ETL workflow

final_report = (
    df.filter(col("Age") > 25)
      .groupBy("Region")
      .agg(
          count("*").alias("Employee_Count"),
          avg("Monthly_Income").alias("Average_Income"),
          sum("Monthly_Income").alias("Total_Income")
      )
)

final_report.show()

+-------+--------------+------------------+------------+
| Region|Employee_Count|    Average_Income|Total_Income|
+-------+--------------+------------------+------------+
|Unknown|             1|           62000.0|       62000|
|  South|             3|54333.333333333336|      163000|
|   East|             2|           75000.0|      150000|
|  North|             3|           60000.0|      180000|
+-------+--------------+------------------+------------+



## Conclusion

Successfully implemented a Spark-based ETL pipeline using PySpark in Google Colab. The project covered data cleaning, transformation, aggregation, and analytical reporting to generate business insights from customer data while demonstrating core data engineering concepts and Spark DataFrame operations.

In [25]:
# Final project completion message

print("=" * 60)
print(" Apache Spark Data Processing Pipeline Executed Successfully ")
print(" Environment : Google Colab")
print(" Framework   : PySpark")
print(" Status      : Completed")
print("=" * 60)

spark.stop()

 Apache Spark Data Processing Pipeline Executed Successfully 
 Environment : Google Colab
 Framework   : PySpark
 Status      : Completed
